# Python API Example - Freight Contracts

This guide is designed to provide an example of how to call the Spark Freight Contracts endpoint
- The path to your client credentials is the only input needed to run this script (just before Section 2)
- This script has been designed to display the raw outputs of requests from the API, and then shows you how to format those outputs to enable easy reading and analysis
 
__N.B. This guide is just for Freight Contracts data. If you're looking for other API data products (such as Freight routes or Netbacks), please refer to their according code example files.__ 


### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint: https://www.sparkcommodities.com/api/lng-freight/contracts.html

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already. Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/request/authentication.html


The code then prints the available prices that are callable from the API, and their corresponding Python ticker names are displayed as a list at the bottom of the Output.

In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

## 2. Listing all available contract tickers

In [ ]:
# Define function for listing contracts from API
def list_contracts(access_token):
    
    content = do_api_get_query(uri="/v1.0/contracts/", access_token=access_token)

    print(">>>> All the contracts you can fetch")
    tickers = []
    for contract in content["data"]:
        print(contract["fullName"])
        tickers.append(contract["id"])

    return tickers

# Fetch all contracts:
tickers = list_contracts(access_token)

print(tickers)

## 3. Latest Price Release

Here we define a function to fetch the latest price release. The available parameters can be found on our API docs:

Freight: https://www.sparkcommodities.com/api/lng-freight/contracts.html

__N.B.:__ This endpoint provides the option to return a JSON or CSV formatted response. Metadata is only available in the JSON format.

In [ ]:
## Defining the function

def fetch_latest_price_releases(access_token, ticker):

    content = do_api_get_query(
        uri="/v1.0/contracts/{}/price-releases/latest/".format(ticker),
        access_token=access_token,
    )

    return content["data"]


## Calling that function and storing the output
data = fetch_latest_price_releases(access_token, 'spark30s')

In [ ]:
# Shows how the raw output is formatted
data

## 4. Historical Price Releases

Here we define a function to fetch the historical price releases. The available parameters can be found on our API docs:

Freight: https://www.sparkcommodities.com/api/lng-freight/contracts.html

__N.B.:__ This endpoint provides the option to return a JSON or CSV formatted response. Metadata is only available in the JSON format.

In [ ]:
def fetch_historical_price_releases(access_token, ticker, limit=4, offset=None):

    print(">>>> Get price releases for {}".format(ticker))

    query_params = "?limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)

    content = do_api_get_query(
        uri="/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params),
        access_token=access_token,
    )

    data = content["data"]

    return data

In [ ]:
### Define which price product you want to retrieve
example_hist_data = fetch_historical_price_releases(access_token, ticker='spark30s', limit=10)

### Formatting into a Pandas DataFrame

The outputted data has several nested lists and dictionaries. If we are aware of what variables we want, we can externally store these values as lists and create a Pandas DataFrame.

The dictionary is then transformed into a Pandas Dataframe for readability and ease of use. 


__N.B.__ This JSON structure is not consistent across all datasets, and so might need to be amended when calling other Spark contracts.

In [ ]:
# Defining the function for storing and formatting the data into a Pandas DataFrame
def store_and_format(dict_hist):
    stored_data = {
        "ReleaseDate": [],
        "Ticker": [],
        "PeriodStart": [],
        "USDperday": [],
        "USDperdayMax": [],
        "USDperdayMin": [],
    }

    for release in dict_hist:

        data_points = release["data"][0]["dataPoints"]

        for data_point in data_points:
            stored_data['Ticker'].append(release["contractId"])
            stored_data['ReleaseDate'].append(release["releaseDate"])

            period_start_at = data_point["deliveryPeriod"]["startAt"]
            stored_data['PeriodStart'].append(period_start_at)

            stored_data['USDperday'].append(data_point["derivedPrices"]["usdPerDay"]["spark"])
            stored_data['USDperdayMin'].append(data_point["derivedPrices"]["usdPerDay"]["sparkMin"])
            stored_data['USDperdayMax'].append(data_point["derivedPrices"]["usdPerDay"]["sparkMax"])
            
    historical_df = pd.DataFrame(stored_data)
        
    historical_df["USDperday"] = pd.to_numeric(historical_df["USDperday"])
    historical_df["USDperdayMax"] = pd.to_numeric(historical_df["USDperdayMax"])
    historical_df["USDperdayMin"] = pd.to_numeric(historical_df["USDperdayMin"])

    historical_df["ReleaseDate"] = pd.to_datetime(historical_df["ReleaseDate"])
        
    return historical_df

# 4. Calling Freight Data

# 4.1. Spot and FFA

In [ ]:
# Calling and storing Spark30S data
spark30s_data = fetch_historical_price_releases(access_token, ticker='spark30s', limit=10)
spark30s = store_and_format(spark30s_data)
spark30s.head()

In [ ]:
# Calling and storing Spark30FFA data
spark30ffa_data = fetch_historical_price_releases(access_token, ticker='spark30ffa-monthly', limit=10)
spark30ffa = store_and_format(spark30ffa_data)
spark30ffa.head()

## 4.2. SparkFFA Balmo

In [ ]:
# Defining the function for storing and formatting the data into a Pandas DataFrame

def balmo_dataframe(my_ticker, limit, offset):
    
    my_dict_hist = fetch_historical_price_releases(access_token, my_ticker, limit=limit, offset=offset)

    stored_data = {
        "ReleaseDate": [],
        "Ticker": [],
        "PeriodStart": [],
        "M+0": [],
        "Balmo":[],
    }

    for release in my_dict_hist:

        data_point = release["data"][0]["dataPoints"][0]
        
        stored_data['ReleaseDate'].append(release["releaseDate"])
        stored_data['Ticker'].append(release["contractId"])

        stored_data['PeriodStart'].append(data_point["deliveryPeriod"]["startAt"])

        stored_data['M+0'].append(data_point["derivedPrices"]["usdPerDay"]["spark"])
        stored_data['Balmo'].append(release['data'][0]['aggregatedData']['balanceOfMonth']['prices']['userPerDay'])
        
    historical_df = pd.DataFrame(stored_data)
    
    historical_df["M+0"] = pd.to_numeric(historical_df["M+0"])
    historical_df["Balmo"] = pd.to_numeric(historical_df["Balmo"])

    historical_df["ReleaseDate"] = pd.to_datetime(historical_df["ReleaseDate"])
    historical_df["PeriodStart"] = pd.to_datetime(historical_df["PeriodStart"])

    historical_df['ContractMonth'] = historical_df["PeriodStart"].dt.strftime('%b-%Y')
        
    return historical_df

In [ ]:
spark30ffa_balmo = balmo_dataframe('spark30ffa-monthly', limit=4, offset=0)
spark30ffa_balmo

# Analytics Gallery
Want to gain market insights using our data?

Take a look at our [Analytics Gallery](https://www.sparkcommodities.com/api/code-examples/analytics-examples.html) on the Spark API website, which includes:

- __Freight Spot Price Seasonality Chart__ - Compare freight spot prices across several years, to understand how the current market compares to historical prices at equivalent periods (e.g. Dec22 vs Dec23 vs Dec24).
